In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Spécifier les options du Sampler

<Accordion>
<AccordionItem title="Versions des packages">

Le code de cette page a été développé avec les prérequis suivants.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Tu peux utiliser des options pour personnaliser la primitive Sampler. Cette section explique comment spécifier les options de la primitive Qiskit Runtime. Bien que l'interface de la méthode `run()` des primitives soit commune à toutes les implémentations, leurs options ne le sont pas. Consulte les références API correspondantes pour obtenir des informations sur les options de [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) et [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## Définir les options du Sampler
Tu peux définir des options lors de l'initialisation du Sampler, après son initialisation, ou mettre à jour les options une fois qu'il a été initialisé. Pour savoir comment utiliser ces techniques, consulte la rubrique [Introduction aux options](/guides/runtime-options-overview#options-precedence).

De plus, tu peux définir la valeur `shots` dans la méthode `run()`, comme décrit dans la section suivante.
<span id="run-method"></span>
### Méthode Run()
Les seules valeurs que tu peux passer à `run()` sont celles définies dans l'interface, c'est-à-dire `shots`. Cela écrase toute valeur définie pour `default_shots` pour l'exécution en cours.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

<Admonition type="note">
Although shots specified in the PUB and in `run` have higher precedence, the job fails if `twirling` is enabled and the product of `num_randomizations` and `shots_per_randomization` is smaller than the `shots` value. In this scenario, `SamplerV2` is unable to allocate the shots among the specified `num_randomizations`.
</Admonition>

Example:

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden
# if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### Cas particuliers
<span id="shots"></span>
#### Shots

La méthode `SamplerV2.run` accepte deux arguments : une liste de PUBs, chacun pouvant spécifier une valeur de shots propre au PUB, et un argument mot-clé shots. Ces valeurs de shots font partie de l'interface d'exécution du Sampler et sont indépendantes des options du Sampler Runtime. Elles ont la priorité sur toute valeur spécifiée dans les options afin de respecter l'abstraction Sampler.

Cependant, si `shots` n'est spécifié ni par un PUB ni dans l'argument mot-clé run (ou s'ils sont tous `None`), alors la valeur de shots provenant des options est utilisée, notamment `default_shots`.

Pour résumer, voici l'ordre de priorité pour spécifier les shots dans le Sampler, pour tout PUB particulier :

1. Si le PUB spécifie des shots, utiliser cette valeur.
2. Si l'argument mot-clé `shots` est spécifié dans `run`, utiliser cette valeur.
4. Si `twirling` est activé (True par défaut), alors le produit de `num_randomizations` et de `shots_per_randomization`, tel que spécifié dans les [options `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options), est utilisé.
5. Si `sampler.options.default_shots` est spécifié, utiliser cette valeur.

Ainsi, si des shots sont spécifiés à tous les endroits possibles, celui ayant la priorité la plus haute (shots spécifiés dans le PUB) est utilisé.

Exemple :